In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


#CONFIG
CSV_PATH   = "/content/Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG = "var_dengue_2025_prediction.png"
VAR_COLS   = ['DENGUE', 'MIN', 'MAX', 'HUMIDITY', 'RAINFALL']
LAG        = 11      # AIC-optimal lag (tested 1–15, AIC minimised at 11)

# 1.  DATA LOADING & CHRONOLOGICAL SORTING
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(
    df['YEAR'].astype(str) + '-' + df['MONTH'].astype(str) + '-01'
)
df = df.sort_values('DATE').reset_index(drop=True)
df.set_index('DATE', inplace=True)
df.index.freq = 'MS'

# 2.  STATIONARITY CHECK & FIRST DIFFERENCING
df_var = df[VAR_COLS].copy()
df_var['DENGUE'] = np.log1p(df_var['DENGUE'])   # log-transform stabilises variance

print("=== ADF Stationarity Tests (levels) ===")
for col in df_var.columns:
    p = adfuller(df_var[col].dropna())[1]
    print(f"  {col:<12}: p={p:.4f}  {'STATIONARY' if p < 0.05 else 'NON-STATIONARY → will difference'}")

df_diff = df_var.diff().dropna()
df_diff.index.freq = 'MS'

print("\n=== ADF After 1st Differencing ===")
for col in df_diff.columns:
    p = adfuller(df_diff[col].dropna())[1]
    print(f"  {col:<12}: p={p:.4f}  {'STATIONARY ✓' if p < 0.05 else 'NON-STATIONARY'}")


# 3.  OPTIMAL LAG SELECTION (on training data only)
train_full = df_diff[:'2024-12-01']

print("\n=== VAR Lag Order Selection ===")
lag_sel = VAR(train_full).select_order(maxlags=15)
print(lag_sel.summary())
aic_lag = lag_sel.selected_orders['aic']
print(f"AIC-optimal lag: {aic_lag}  (using LAG={LAG})")

# 4.  WALK-FORWARD PREDICTION  (Jan – Oct 2025)
pred_dates = pd.date_range('2025-01-01', periods=10, freq='MS')
y_pred = []

print("\n=== Walk-Forward Predictions ===")
for step, target_date in enumerate(pred_dates):
    # Train on everything before target month
    cutoff_train = target_date - pd.DateOffset(months=1)
    train_wf     = df_diff[:cutoff_train]

    # Fit VAR(LAG) on rolling window
    model_wf  = VAR(train_wf)
    fitted_wf = model_wf.fit(LAG)

    # 1-step forecast in differenced space
    fc_diff = fitted_wf.forecast(train_wf.values[-LAG:], steps=1)[0]

    # Invert differencing using last known log-level
    last_log_level = df_var.loc[cutoff_train]
    log_level_pred = last_log_level.values + fc_diff

    # Invert log transform for DENGUE (index 0)
    cases_pred = max(0.0, np.expm1(log_level_pred[0]))
    y_pred.append(cases_pred)

y_pred = np.array(y_pred)
y_true = df['DENGUE'].loc['2025-01-01':].values.astype(float)

# 5.  CONFIDENCE BAND  (±1.96 σ of training residuals, in log space)
final_model  = VAR(train_full).fit(LAG)
resid_std    = final_model.resid['DENGUE'].std()
log_pred     = np.log1p(y_pred)
lb = np.clip(np.expm1(log_pred - 1.96 * resid_std), 0, None)
ub =         np.expm1(log_pred + 1.96 * resid_std)


# 6.  PERFORMANCE METRICS
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true) * 100)

print(f"\n=== VAR WALK-FORWARD PERFORMANCE  (Jan – Oct 2025) ===")
print(f"  MAE  : {mae:>10,.0f} cases")
print(f"  RMSE : {rmse:>10,.0f} cases")
print(f"  R²   : {r2:>10.4f}")
print(f"  MAPE : {mape:>10.1f} %")

print(f"\n{'Month':<12} {'Predicted':>12} {'Observed':>10} {'Error %':>9}")
print("-" * 48)
for d, p, o in zip(pred_dates, y_pred, y_true):
    pct = abs(p - o) / o * 100
    print(f"{d.strftime('%b %Y'):<12} {int(p):>12,} {int(o):>10,} {pct:>8.1f}%")

print(f"\n{'Final model summary':}")
print(final_model.summary())


# 7.  PUBLICATION-QUALITY 4-PANEL FIGURE
BG, PANEL, GRID      = '#0d1117', '#161b22', '#21262d'
BORDER, WHITE, MUTED = '#30363d', '#e6edf3', '#8b949e'
BLUE, ORANGE, GREEN, YELLOW = '#58a6ff', '#f78166', '#3fb950', '#d29922'

def style_ax(ax, title, ylabel='Dengue Cases'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=9)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

fig = plt.figure(figsize=(15, 14), dpi=150)
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(3, 2, hspace=0.44, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])

#Panel 1: Full timeline 2008-2025
style_ax(ax1, '▸  Full Timeline: Dengue Cases in Bangladesh  (2008 – 2025)')
hist = df['DENGUE'].loc[:'2024-12-01']
ax1.plot(hist.index, hist.values, color=MUTED, lw=1.3, alpha=0.55,
         label='Historical Data  (2008–2024)')
ax1.plot(pred_dates, y_true, color=BLUE, lw=2, marker='o', ms=4,
         label='Observed 2025  (Jan–Oct)')
ax1.plot(pred_dates, y_pred, color=ORANGE, lw=2, ls='--', marker='x', ms=5,
         label='VAR Walk-Forward Forecast')
ax1.fill_between(pred_dates, lb, ub, color=ORANGE, alpha=0.15, label='95 % CI')
ax1.axvline(pd.Timestamp('2025-01-01'), color=YELLOW, lw=1.2, ls=':', alpha=0.85,
            label='2025 Start')
ax1.set_xlabel('Year', color=MUTED, fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE,
           framealpha=0.9, edgecolor=BORDER, loc='upper left', ncol=3)

#Panel 2: 2025 zoom with per-month error %
style_ax(ax2, '▸  2025 Outbreak Prediction vs Observed  (Jan – Oct) — Walk-Forward VAR(11)')
ax2.plot(pred_dates, y_true, color=BLUE,   lw=2.3, marker='o', ms=6, zorder=4,
         label='Observed Cases  (DGHS)')
ax2.plot(pred_dates, y_pred, color=ORANGE, lw=2.3, ls='--', marker='x', ms=7, zorder=4,
         label='VAR Forecast')
ax2.fill_between(pred_dates, lb, ub, color=ORANGE, alpha=0.18, label='95 % CI')

# Error % label above each prediction point
for d, p, o in zip(pred_dates, y_pred, y_true):
    pct   = abs(p - o) / o * 100
    clr   = GREEN if pct <= 15 else (YELLOW if pct <= 40 else ORANGE)
    ax2.text(d, max(p, o) + max(y_true) * 0.022,
             f'{pct:.0f}%', ha='center', va='bottom',
             fontsize=7.5, color=clr, fontweight='bold')

obs_pk  = np.argmax(y_true)
pred_pk = np.argmax(y_pred)
ax2.annotate(f'Observed Peak\n{int(y_true[obs_pk]):,}',
             xy=(pred_dates[obs_pk], y_true[obs_pk]),
             xytext=(pred_dates[max(obs_pk-3, 0)], y_true[obs_pk] * 0.76),
             color=YELLOW, fontsize=9,
             arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2))
ax2.annotate(f'Predicted Peak\n{int(y_pred[pred_pk]):,}',
             xy=(pred_dates[pred_pk], y_pred[pred_pk]),
             xytext=(pred_dates[max(pred_pk-3, 0)], y_pred[pred_pk] * 1.25),
             color=ORANGE, fontsize=9,
             arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2))

ax2.text(0.99, 0.04,
         f'MAE: {mae:,.0f}  |  RMSE: {rmse:,.0f}  |  R²: {r2:.3f}  |  MAPE: {mape:.1f}%',
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=9.5, color=WHITE,
         bbox=dict(boxstyle='round,pad=0.5', facecolor=BG, edgecolor=BORDER, alpha=0.92))
ax2.set_xlabel('Month', color=MUTED, fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE,
           framealpha=0.9, edgecolor=BORDER, loc='upper left')

#Panel 3: Monthly bar comparison
style_ax(ax3, '▸  Monthly Bar: Observed vs Predicted')
months_lbl = [d.strftime('%b') for d in pred_dates]
x, w = np.arange(10), 0.38
ax3.bar(x - w/2, y_true, w, color=BLUE,   alpha=0.85, label='Observed',  zorder=3)
ax3.bar(x + w/2, y_pred, w, color=ORANGE, alpha=0.85, label='Predicted', zorder=3)
ax3.set_xticks(x)
ax3.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax3.set_xlabel('Month (2025)', color=MUTED, fontsize=9)
ax3.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE,
           framealpha=0.9, edgecolor=BORDER)

#Panel 4: Monthly error % bar
style_ax(ax4, '▸  Monthly Error %  (per-month MAPE)', ylabel='Error %')
pct_errs  = [abs(p - o) / o * 100 for p, o in zip(y_pred, y_true)]
clrs_e    = [GREEN if e <= 15 else (YELLOW if e <= 40 else ORANGE) for e in pct_errs]
bars_e    = ax4.bar(range(10), pct_errs, color=clrs_e, alpha=0.88, zorder=3)
ax4.axhline(15, color=GREEN,  lw=1.2, ls='--', alpha=0.7, label='15 % (good)')
ax4.axhline(40, color=YELLOW, lw=1.2, ls='--', alpha=0.7, label='40 % (acceptable)')
for b, e in zip(bars_e, pct_errs):
    ax4.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6,
             f'{e:.0f}%', ha='center', fontsize=8, color=WHITE, fontweight='bold')
ax4.set_xticks(range(10))
ax4.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax4.set_xlabel('Month (2025)', color=MUTED, fontsize=9)
ax4.text(0.98, 0.97, 'Green ≤ 15 %\nYellow ≤ 40 %\nOrange > 40 %',
         transform=ax4.transAxes, ha='right', va='top',
         fontsize=8, color=MUTED, linespacing=1.6)
ax4.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE,
           framealpha=0.9, edgecolor=BORDER, loc='upper left')

fig.suptitle(
    'VAR (Vector Autoregression) — Walk-Forward Dengue Outbreak Prediction  |  Bangladesh 2025  (Jan – Oct)\n'
    'Variables: DENGUE · MIN · MAX · HUMIDITY · RAINFALL  |  Optimal Lag = 11  (AIC)  |  Monthly Refit Strategy',
    color=WHITE, fontsize=11.5, fontweight='bold', y=0.999,
)

plt.savefig(OUTPUT_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"\nFigure saved → {OUTPUT_PNG}")
plt.show()

=== ADF Stationarity Tests (levels) ===
  DENGUE      : p=0.7266  NON-STATIONARY → will difference
  MIN         : p=0.9735  NON-STATIONARY → will difference
  MAX         : p=0.1638  NON-STATIONARY → will difference
  HUMIDITY    : p=0.0363  STATIONARY
  RAINFALL    : p=0.0069  STATIONARY

=== ADF After 1st Differencing ===
  DENGUE      : p=0.0000  STATIONARY ✓
  MIN         : p=0.0000  STATIONARY ✓
  MAX         : p=0.0000  STATIONARY ✓
  HUMIDITY    : p=0.0000  STATIONARY ✓
  RAINFALL    : p=0.0000  STATIONARY ✓

=== VAR Lag Order Selection ===
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0        14.88       14.97   2.901e+06       14.92
1        13.46       13.98   7.033e+05       13.67
2        12.85       13.80   3.829e+05       13.24
3        12.28      13.66*   2.155e+05       12.84
4        11.89       13.70   1.462e+05       12.62
5        11.78       14.02   1.324e+0